# IO + Preprocessing Smoke Test

Loads one subject's data, runs preprocessing, and plots results for visual inspection.
All expected values confirmed from MATLAB pipeline.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from gait_ml.config import DEFAULT_LAB_CONFIG as cfg
from gait_ml.io import (
    load_marker_tsv,
    load_force_tsv,
    load_subject_weight_newtons,
    load_subject,
    detect_running_belt,
)
from gait_ml.preprocessing import (
    preprocess_markers,
    preprocess_forces,
    detect_missing_markers,
)

RAW = Path('../data/raw')
DEMO = RAW / 'd_subjectData.csv'
SUBJECT = 'FS6'

## 1. Body weight

In [ ]:
bw_n = load_subject_weight_newtons(SUBJECT, RAW)
bw_kg = bw_n / 9.81
print(f'Body weight: {bw_n:.1f} N  ({bw_kg:.1f} kg)')
# Sanity check: should be a plausible human weight (400–1000 N)

## 2. Marker TSV loading

In [ ]:
trial_path = RAW / f'{SUBJECT}_WalkingPreferred1.tsv'
markers_raw = load_marker_tsv(trial_path)

print(f'Shape: {markers_raw.shape}  (frames × columns)')
print(f'Duration: {markers_raw["Time"].iloc[-1]:.2f} s  @ {cfg.acquisition.kinematic_sample_rate_hz} Hz')
print(f'Columns (first 10): {list(markers_raw.columns[:10])}')
print(f'NaN count (marker cols): {markers_raw.iloc[:, 2:].isna().sum().sum()}')

In [ ]:
# Confirm coordinate system: CLAV Y should be ~1200 mm (standing height)
# LASIS/RASIS should differ mainly in Z (mediolateral ~200–250 mm apart)
print('CLAVY mean:', markers_raw['CLAVY'].mean().round(1), 'mm  (expect ~1200)')
if 'LASISZ' in markers_raw.columns and 'RASISZ' in markers_raw.columns:
    pelvis_sep = (markers_raw['LASISZ'] - markers_raw['RASISZ']).abs().mean()
    print(f'LASIS-RASIS Z separation: {pelvis_sep:.1f} mm  (expect ~200–250)')

## 3. Missing marker detection

In [ ]:
marker_cols = list(markers_raw.columns[2:])

# Group columns by marker name (strip trailing X/Y/Z)
# e.g. CLAVX, CLAVY, CLAVZ → CLAV
def marker_name(col):
    for axis in ('X', 'Y', 'Z'):
        if col.endswith(axis):
            return col[:-1]
    return col

marker_names = sorted(set(marker_name(c) for c in marker_cols))

# A marker is missing at a frame when all 3 of its axes are NaN
gap_frames = {}   # marker → bool array (True = missing)
for m in marker_names:
    axes = [c for c in marker_cols if marker_name(c) == m]
    gap_frames[m] = markers_raw[axes].isna().all(axis=1).to_numpy()

# Summary table: markers that have any gaps
rows = []
n_frames = len(markers_raw)
fs = cfg.acquisition.kinematic_sample_rate_hz
for m, mask in gap_frames.items():
    n = mask.sum()
    if n > 0:
        rows.append({'Marker': m, 'Missing frames': n,
                     'Missing %': round(100 * n / n_frames, 1),
                     'Duration (s)': round(n / fs, 3)})

if rows:
    gap_df = pd.DataFrame(rows).sort_values('Missing frames', ascending=False)
    print(f'{len(rows)} marker(s) with gaps (out of {len(marker_names)} total):\n')
    print(gap_df.to_string(index=False))
else:
    print('No missing markers detected.')

In [ ]:
# Gap timeline — one row per marker with gaps, black = missing
if rows:
    gap_markers = gap_df['Marker'].tolist()
    t = markers_raw['Time'].to_numpy()

    fig, ax = plt.subplots(figsize=(12, max(3, len(gap_markers) * 0.35)))
    for i, m in enumerate(gap_markers):
        mask = gap_frames[m]
        ax.fill_between(t, i, i + 0.8, where=mask, color='black', linewidth=0)

    ax.set_yticks(np.arange(len(gap_markers)) + 0.4)
    ax.set_yticklabels(gap_markers, fontsize=8)
    ax.set_xlabel('Time (s)')
    ax.set_title(f'{SUBJECT} WalkingPreferred1 — marker gaps (black = occluded)')
    ax.set_xlim(t[0], t[-1])
    plt.tight_layout()
    plt.show()

In [ ]:
from gait_ml.preprocessing import fill_marker_gaps

marker_cols = list(markers_raw.columns[2:])
arr_raw = markers_raw[marker_cols].to_numpy(dtype=float)
arr_filled = fill_marker_gaps(arr_raw)

# Pick markers that actually have gaps — use gap_df from section 3
if 'gap_df' not in dir() or gap_df.empty:
    print('Run section 3 first to populate gap_df.')
else:
    show_markers = gap_df['Marker'].head(6).tolist()
    n = len(show_markers)

    fig, axes = plt.subplots(n, 1, figsize=(13, 3.5 * n), sharex=True)
    if n == 1:
        axes = [axes]

    t = markers_raw['Time'].to_numpy()

    for ax, m in zip(axes, show_markers):
        for suffix, color in [('X', 'C2'), ('Z', 'C4'), ('Y', 'C0')]:
            col = m + suffix
            if col not in marker_cols:
                continue
            ci = marker_cols.index(col)
            raw_vals    = arr_raw[:, ci].copy()
            filled_vals = arr_filled[:, ci].copy()
            # faded raw (NaN produces line breaks), solid filled
            ax.plot(t, raw_vals,    lw=1.0, color=color, alpha=0.35,
                    label=f'raw ({suffix})')
            ax.plot(t, filled_vals, lw=1.2, color=color,
                    label=f'filled ({suffix})')

        # Shade gap regions after lines are drawn so ylim is correct
        ax.autoscale(axis='y')
        ylo, yhi = ax.get_ylim()
        mask = gap_frames[m]
        ax.fill_between(t, ylo, yhi, where=mask,
                        color='red', alpha=0.12, zorder=0, label='gap')
        ax.set_ylim(ylo, yhi)

        n_gaps = gap_df.loc[gap_df['Marker'] == m, 'Missing frames'].values[0]
        pct    = gap_df.loc[gap_df['Marker'] == m, 'Missing %'].values[0]
        ax.set_title(f'{m}  ({n_gaps} missing frames, {pct}%)', fontsize=9)
        ax.set_ylabel('mm')
        ax.legend(fontsize=7, loc='upper right', ncol=4)

    axes[-1].set_xlabel('Time (s)')
    fig.suptitle(f'{SUBJECT} WalkingPreferred1 — gap-fill: faded=raw, solid=filled, red=gap region',
                 y=1.01, fontsize=10)
    plt.tight_layout()
    plt.show()

## 5. Force TSV loading

In [ ]:
force_l = load_force_tsv(RAW / f'{SUBJECT}_WalkingPreferred1_f_4.tsv')
force_r = load_force_tsv(RAW / f'{SUBJECT}_WalkingPreferred1_f_5.tsv')

print(f'Left belt shape:  {force_l.shape}')
print(f'Right belt shape: {force_r.shape}')
print(f'Columns: {list(force_l.columns)}')
print(f'Expected ratio GRF/kin frames: {force_l.shape[0] / markers_raw.shape[0]:.2f}  (expect 7.0)')
print(f'Left  Force_Z max: {force_l["Force_Z"].max():.1f} N')
print(f'Right Force_Z max: {force_r["Force_Z"].max():.1f} N')

## 6. Force preprocessing: filter

In [ ]:
force_l_proc = preprocess_forces(force_l)
force_r_proc = preprocess_forces(force_r)

t_f = force_l['TIME'].to_numpy()

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, col, label in zip(axes, ['Force_X', 'Force_Y', 'Force_Z'],
                           ['Fx (N)', 'Fy (N)', 'Fz (N)']):
    ax.plot(t_f, force_l[col].to_numpy(), lw=0.5, alpha=0.6, label='raw L')
    ax.plot(t_f, force_l_proc[col].to_numpy(), lw=1.0, label='filtered L (8 Hz)')
    ax.set_ylabel(label)
    ax.legend(fontsize=8)

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Left belt GRF — raw vs filtered')
plt.tight_layout()
plt.show()

## 7. Both belts — walking

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t_f, force_l_proc['Force_Z'].to_numpy(), label='Left belt (f_4)', lw=0.8)
ax.plot(t_f, force_r_proc['Force_Z'].to_numpy(), label='Right belt (f_5)', lw=0.8)
ax.axhline(bw_n, color='k', ls='--', lw=0.8, label=f'Body weight ({bw_n:.0f} N)')
ax.set_ylabel('Fz (N)')
ax.set_xlabel('Time (s)')
ax.set_title('Vertical GRF — WalkingPreferred trial 1 (both belts)')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Running belt detection

In [ ]:
run_conditions = ['RunningPreDetermined', 'RunningFroudeA', 'RunningFroudeB']

for cond in run_conditions:
    for trial in range(1, 4):
        stem = RAW / f'{SUBJECT}_{cond}{trial}'
        fl_path = Path(str(stem) + '_f_4.tsv')
        fr_path = Path(str(stem) + '_f_5.tsv')
        if not fl_path.exists():
            continue
        fl = load_force_tsv(fl_path)
        fr = load_force_tsv(fr_path)
        belt = detect_running_belt(fl, fr, bw_n)
        peak_l = fl['Force_Z'].max()
        peak_r = fr['Force_Z'].max()
        print(f'{cond} trial {trial}: belt={belt}  '
              f'peak_L={peak_l:.0f} N  peak_R={peak_r:.0f} N  BW={bw_n:.0f} N')

## 9. load_subject — all trials for one subject

In [ ]:
subject = load_subject(SUBJECT, RAW, DEMO)

print(f'Subject:       {subject.subject_id}')
print(f'Age/sex:       {subject.meta.age} y, {subject.meta.sex}')
print(f'Body weight:   {subject.body_weight_n:.1f} N  ({subject.body_weight_kg:.1f} kg)')
print(f'Height:        {subject.meta.height_cm} cm')
print(f'Leg length:    R={subject.meta.leg_length_r_cm} cm  L={subject.meta.leg_length_l_cm} cm')
print(f'Speeds:        {subject.meta.speeds}')
print()
print(f'Conditions loaded: {list(subject.trials.keys())}')
print(f'Walk trials: {len(subject.walk_trials())}  Run trials: {len(subject.run_trials())}')
print()
for cond, trials in subject.trials.items():
    shapes = [t.markers.shape for t in trials]
    belts  = [t.belt or '-' for t in trials]
    print(f'  {cond}: {len(trials)} trials  shapes={shapes}  belts={belts}')
subject.unload()  # release cached DataFrames